In [15]:
# spark.stop()

In [2]:
import sys
import os
from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


# ============================================================
# PROJECT PATH
# Notebook location:
# capstone/product_analysis/notebooks/
#
# Therefore, parent directory is:
# capstone/product_analysis/
# ============================================================

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


# ============================================================
# LANDING SOURCE
# ============================================================

LANDING_ROOT = Path(
    "/mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002"
)

SOURCE_FILE = str(
    LANDING_ROOT / "sales" / "order-lines" / "malformed.csv"
)

BATCH_ID = "api-00002-malformed-03"
DATASET_NAME = "sales_order_lines"


# ============================================================
# EXPECTED SCHEMA FILE
# ============================================================

SCHEMA_FILE = (
    PROJECT_ROOT
    / "schemas"
    / "sales_order_lines_schema.json"
)


# ============================================================
# HDFS PATHS
# ============================================================

BRONZE_PATH = (
    "hdfs:///capstone/warehouse/bronze/"
    "sales_order_lines"
)

# Iceberg warehouse location used by the datalake catalog
WAREHOUSE = "hdfs:///capstone/warehouse"


# ============================================================
# ICEBERG AUDIT TABLE
# ============================================================

CATALOG = "datalake"
AUDIT_NAMESPACE = "operations"

AUDIT_TABLE = (
    f"{CATALOG}.{AUDIT_NAMESPACE}.schema_drift"
)


# ============================================================
# NOTEBOOK / SPARK APPLICATION NAME
# ============================================================

NOTEBOOK_NAME = "sales_order_lines_landing_to_bronze"


# ============================================================
# BASIC FILE VALIDATION
# ============================================================

print("Source file:", SOURCE_FILE)
print("Schema file:", SCHEMA_FILE)
print("Bronze path:", BRONZE_PATH)
print("Audit table:", AUDIT_TABLE)
print("Batch ID:", BATCH_ID)

print("\nLocal file checks:")
print("Source file exists:", Path(SOURCE_FILE).is_file())
print("Schema file exists:", SCHEMA_FILE.is_file())

Project root: /mnt/c/santhosh/capstone/product_analysis
Source file: /mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002/sales/order-lines/malformed.csv
Schema file: /mnt/c/santhosh/capstone/product_analysis/schemas/sales_order_lines_schema.json
Bronze path: hdfs:///capstone/warehouse/bronze/sales_order_lines
Audit table: datalake.operations.schema_drift
Batch ID: api-00002-malformed-03

Local file checks:
Source file exists: True
Schema file exists: True


In [3]:
# Used to relaod the schema_dirft.py module 
import importlib
import data_profilers.schema_drift as schema_drift

schema_drift = importlib.reload(schema_drift)

process_landing_csv = schema_drift.process_landing_csv

print("Reloaded:", schema_drift.__file__)

Reloaded: /mnt/c/santhosh/capstone/product_analysis/data_profilers/schema_drift.py


In [4]:
builder = (
    SparkSession.builder
    .appName(NOTEBOOK_NAME)
    .master(os.environ.get("SPARK_MASTER", "local[*]"))

    # Iceberg Spark extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions."
        "IcebergSparkSessionExtensions"
    )

    # Iceberg catalog configuration
    .config(
        f"spark.sql.catalog.{CATALOG}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{CATALOG}.type",
        "hadoop"
    )
    .config(
        f"spark.sql.catalog.{CATALOG}.warehouse",
        WAREHOUSE
    )

    # General Spark settings
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.caseSensitive", "true")
    .config("spark.sql.shuffle.partitions", "2")

    # Required for reliable _corrupt_record filtering
    .config(
        "spark.sql.csv.parser.columnPruning.enabled",
        "false"
    )
)

# Add Iceberg runtime JAR when provided through environment variable
iceberg_jar = os.environ.get("ICEBERG_SPARK_RUNTIME_JAR")

if iceberg_jar:
    builder = builder.config("spark.jars", iceberg_jar)
    print("Using Iceberg runtime JAR:", iceberg_jar)
else:
    print(
        "ICEBERG_SPARK_RUNTIME_JAR is not explicitly set. "
        "Assuming Iceberg dependencies are already available."
    )

spark = builder.getOrCreate()

spark.sparkContext.setLogLevel("WARN")


# ============================================================
# VERIFY SPARK AND ICEBERG CONFIGURATION
# ============================================================

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

print(
    "Iceberg catalog:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}"
    )
)

print(
    "Iceberg catalog type:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}.type"
    )
)

print(
    "Iceberg warehouse:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}.warehouse"
    )
)

print(
    "CSV column pruning enabled:",
    spark.conf.get(
        "spark.sql.csv.parser.columnPruning.enabled"
    )
)

ICEBERG_SPARK_RUNTIME_JAR is not explicitly set. Assuming Iceberg dependencies are already available.


26/10/11 11:52:46 WARN Utils: Your hostname, 5CD6260W0K resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 11:52:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 11:52:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 11:52:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.9
Spark master: local[*]
Iceberg catalog: org.apache.iceberg.spark.SparkCatalog
Iceberg catalog type: hadoop
Iceberg warehouse: hdfs:///capstone/warehouse
CSV column pruning enabled: false


In [5]:
import json


# ============================================================
# LOAD SCHEMA JSON
# ============================================================

with open(SCHEMA_FILE, "r", encoding="utf-8") as schema_file:
    schema_definition = json.load(schema_file)

print("Schema definition:")
print(schema_definition)


# ============================================================
# MAP JSON DATATYPE NAMES TO SPARK DATATYPES
# ============================================================

SPARK_TYPE_MAPPING = {
    "string": T.StringType(),
    "integer": T.IntegerType(),
    "int": T.IntegerType(),
    "long": T.LongType(),
    "double": T.DoubleType(),
    "float": T.FloatType(),
    "boolean": T.BooleanType(),
    "date": T.DateType(),
    "timestamp": T.TimestampType(),
}


# ============================================================
# CREATE SPARK STRUCTTYPE
# ============================================================

schema_fields = []

for column_name, datatype_name in schema_definition.items():
    normalized_type = datatype_name.strip().lower()

    if normalized_type not in SPARK_TYPE_MAPPING:
        raise ValueError(
            f"Unsupported datatype '{datatype_name}' "
            f"for column '{column_name}'"
        )

    schema_fields.append(
        T.StructField(
            column_name,
            SPARK_TYPE_MAPPING[normalized_type],
            True
        )
    )

expected_schema = T.StructType(schema_fields)


# ============================================================
# DISPLAY EXPECTED SCHEMA
# ============================================================

print("\nExpected Spark schema:")
print(expected_schema.simpleString())

print("\nExpected schema fields:")

for field in expected_schema.fields:
    print(
        f"{field.name:<20} "
        f"{field.dataType.simpleString():<12} "
        f"nullable={field.nullable}"
    )

Schema definition:
{'id': 'long', 'order_id': 'string', 'product_id': 'string', 'product_uom_qty': 'double', 'price_unit': 'double', 'discount': 'double', 'price_subtotal': 'double', 'create_date': 'timestamp', 'write_date': 'timestamp', 'qty_delivered': 'double', 'qty_invoiced': 'double'}

Expected Spark schema:
struct<id:bigint,order_id:string,product_id:string,product_uom_qty:double,price_unit:double,discount:double,price_subtotal:double,create_date:timestamp,write_date:timestamp,qty_delivered:double,qty_invoiced:double>

Expected schema fields:
id                   bigint       nullable=True
order_id             string       nullable=True
product_id           string       nullable=True
product_uom_qty      double       nullable=True
price_unit           double       nullable=True
discount             double       nullable=True
price_subtotal       double       nullable=True
create_date          timestamp    nullable=True
write_date           timestamp    nullable=True
qty_delivered

In [6]:
import inspect

from data_profilers.schema_drift import process_landing_csv


# ============================================================
# VERIFY IMPORT LOCATION
# ============================================================

template_file = inspect.getfile(process_landing_csv)

print(
    "Template module loaded successfully."
)

print(
    "Imported from:",
    template_file
)

print(
    "Expected location:",
    PROJECT_ROOT
    / "data_profilers"
    / "schema_drift.py"
)


# ============================================================
# VERIFY FUNCTION SIGNATURE
# ============================================================

print(
    "\nFunction signature:"
)

print(
    inspect.signature(
        process_landing_csv
    )
)

Template module loaded successfully.
Imported from: /mnt/c/santhosh/capstone/product_analysis/data_profilers/schema_drift.py
Expected location: /mnt/c/santhosh/capstone/product_analysis/data_profilers/schema_drift.py

Function signature:
(spark, *, landing_path, bronze_path, expected_schema, dataset_name, audit_table='datalake.operations.schema_drift', batch_id=None, reader_options=None, write_mode='append')


In [7]:
# ============================================================
# VALIDATE LOCAL SOURCE FILE
# ============================================================

source_path = Path(SOURCE_FILE)

if not source_path.exists():
    raise FileNotFoundError(
        f"Source CSV does not exist: {SOURCE_FILE}"
    )

if not source_path.is_file():
    raise ValueError(
        f"Source path is not a file: {SOURCE_FILE}"
    )

if source_path.stat().st_size == 0:
    raise ValueError(
        f"Source CSV is empty: {SOURCE_FILE}"
    )

print("Source file validation successful.")
print("Source file:", source_path)
print("File size:", source_path.stat().st_size, "bytes")


# ============================================================
# READ HEADER AND PREVIEW RAW RECORDS
# ============================================================

with open(
    source_path,
    mode="r",
    encoding="utf-8-sig",
    errors="replace"
) as source_file:

    header = source_file.readline().strip()

    preview_rows = []

    for _ in range(5):
        row = source_file.readline()

        if not row:
            break

        preview_rows.append(row.rstrip("\n"))


print("\nCSV header:")
print(header)

print("\nRaw record preview:")

for row_number, row in enumerate(
    preview_rows,
    start=1
):
    print(f"{row_number}: {row}")


# ============================================================
# COMPARE HEADER COLUMNS WITH EXPECTED SCHEMA
# ============================================================

import csv

header_columns = next(
    csv.reader([header])
)

expected_columns = [
    field.name
    for field in expected_schema.fields
]

added_columns = [
    column
    for column in header_columns
    if column not in expected_columns
]

missing_columns = [
    column
    for column in expected_columns
    if column not in header_columns
]


print("\nHeader columns:")
print(header_columns)

print("\nExpected columns:")
print(expected_columns)

print("\nAdded columns:")
print(added_columns)

print("\nMissing columns:")
print(missing_columns)


if not added_columns and not missing_columns:
    print(
        "\nHeader-level schema check: MATCH"
    )
else:
    print(
        "\nHeader-level schema check: DRIFT DETECTED"
    )

Source file validation successful.
Source file: /mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002/sales/order-lines/malformed.csv
File size: 928 bytes

CSV header:
id,order_id,product_id,product_uom_qty,price_unit,discount,price_subtotal,create_date,write_date,qty_delivered,qty_invoiced

Raw record preview:
1: 1000001,"[100000014, ""S00014""]","[500003538, ""[85123A] WHITE HANGING HEART T-LIGHT HOLDER""]",6.0,2.55,0.0,15.3,2026-10-07 17:50:27,2026-10-07 17:50:39,6.0,6.0
2: 1000002,"[100000014, ""S00014""]","[500002796, ""[71053] WHITE METAL LANTERN""]",6.0,3.39,10.0,18.31,2026-10-07 17:50:27,2026-10-07 17:50:39,6.0,6.0
3: 1000003,"[100000014, ""S00014""]","[500003046, ""[84406B] CREAM CUPID HEARTS COAT HANGER""]",8.0,2.75,10.0,19.8,2026-10-07 17:50:27,2026-10-07 17:50:39,8.0,8.0
4: 1000004,"[100000014, ""S00014""]","[500002987, ""[84029G] KNITTED UNION FLAG HOT WATER BOTTLE""]",6.0,3.39,15.0,17.29,"2026-10-07 17:50:27,2026-10-07 17:50:39,6.0,6.0
5: 1000005,"[100000014, ""S00014""]",

In [8]:
# ============================================================
# CONVERT LOCAL WSL PATH TO SPARK FILE URI
# ============================================================

SOURCE_FILE_LOCAL = Path(
    "/mnt/c/data/odoo_ecomm/release-csv/api/csv/"
    "api-00002/sales/order-lines/malformed.csv"
).resolve()

if not SOURCE_FILE_LOCAL.is_file():
    raise FileNotFoundError(
        f"Source file not found: {SOURCE_FILE_LOCAL}"
    )

# Convert:
# /mnt/c/data/.../file.csv
#
# Into:
# file:///mnt/c/data/.../file.csv
SOURCE_FILE = SOURCE_FILE_LOCAL.as_uri()

print("Local source path:", SOURCE_FILE_LOCAL)
print("Spark source URI:", SOURCE_FILE)

Local source path: /mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002/sales/order-lines/malformed.csv
Spark source URI: file:///mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002/sales/order-lines/malformed.csv


In [9]:
# ============================================================
# RUN LANDING-TO-BRONZE INGESTION
# ============================================================

ingestion_result = process_landing_csv(
    spark=spark,

    # Source CSV file in the Landing zone
    landing_path=SOURCE_FILE,

    # HDFS destination for good records
    bronze_path=BRONZE_PATH,

    # Expected schema loaded from JSON
    expected_schema=expected_schema,

    dataset_name=DATASET_NAME,

    # Iceberg table for schema drift, corrupt records,
    # reconciliation, errors, and batch summaries
    audit_table=AUDIT_TABLE,

    batch_id=BATCH_ID,

    reader_options={
        "header": "true",
        "delimiter": ",",
        "quote": '"',
        "escape": '"',
        "multiLine": "true",
        "encoding": "UTF-8",
        "timestampFormat": "yyyy-MM-dd HH:mm:ss",
    },

    # Good records are appended to a batch-specific HDFS path
    write_mode="append",
)


# ============================================================
# DISPLAY RETURNED RESULT
# ============================================================

print("\nLanding-to-Bronze processing completed.")

print("Run ID:", ingestion_result["run_id"])
print("Batch ID:", ingestion_result["batch_id"])
print("Dataset:", ingestion_result["dataset_name"])
print("Status:", ingestion_result["status"])
print("Source count:", ingestion_result["source_count"])
print("Good count:", ingestion_result["good_count"])
print("Bad count:", ingestion_result["bad_count"])

print("\nFile-level results:")

for file_result in ingestion_result["files"]:
    print("-" * 70)

    for key, value in file_result.items():
        print(f"{key}: {value}")


Landing-to-Bronze processing completed.
Run ID: a9f90d91-19c1-40d6-82ed-5ab81ca462f7
Batch ID: api-00002-malformed-03
Dataset: sales_order_lines
Status: SUCCESS
Source count: 4
Good count: 3
Bad count: 1

File-level results:
----------------------------------------------------------------------
source_file: file:/mnt/c/data/odoo_ecomm/release-csv/api/csv/api-00002/sales/order-lines/malformed.csv
source_count: 4
good_count: 3
bad_count: 1
drift_detected: True
reconciliation_status: RECONCILED


In [10]:
# ============================================================
# INSPECT SCHEMA DRIFT FOR THE CURRENT RUN
# ============================================================

CURRENT_RUN_ID = ingestion_result["run_id"]

schema_drift_audit_df = (
    spark.table(AUDIT_TABLE)
    .filter(
        (F.col("run_id") == CURRENT_RUN_ID)
        & (F.col("event_type") == "SCHEMA_DRIFT")
    )
    .select(
        "run_id",
        "batch_id",
        "dataset_name",
        "source_file",
        "status",
        "expected_schema_json",
        "actual_schema_json",
        "drift_details_json",
        "source_count",
        "good_count",
        "bad_count",
        "created_at"
    )
)

print("Schema drift audit for run:", CURRENT_RUN_ID)

schema_drift_audit_df.show(
    truncate=False,
    vertical=True
)

drift_rows = schema_drift_audit_df.select(
    "drift_details_json"
).collect()

for row in drift_rows:
    if row["drift_details_json"]:
        drift_details = json.loads(
            row["drift_details_json"]
        )

        print("\nSchema drift detected:")
        print(
            json.dumps(
                drift_details,
                indent=4
            )
        )

Schema drift audit for run: a9f90d91-19c1-40d6-82ed-5ab81ca462f7
-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 run_id               | a9f90d91-19c1-40d6-82ed-5ab81ca462f7                                                                  

In [11]:
# Verifying that all the good records are written to bronze 

# ============================================================
# BUILD THE BATCH OUTPUT PATH
# ============================================================

BRONZE_BATCH_PATH = (
    f"{BRONZE_PATH}/"
    f"batch_id={BATCH_ID}/"
    "file_00000"
)

print("Bronze batch path:", BRONZE_BATCH_PATH)


# ============================================================
# READ GOOD RECORDS FROM BRONZE HDFS
# ============================================================

bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(BRONZE_BATCH_PATH)
)


# ============================================================
# DISPLAY BRONZE INFORMATION
# ============================================================

bronze_count = bronze_df.count()

print("\nBronze record count:", bronze_count)

print("\nBronze schema:")
bronze_df.printSchema()

print("\nBronze data preview:")

bronze_df.show(
    5,
    truncate=False
)


# ============================================================
# VERIFY RECORD COUNT
# ============================================================

expected_good_count = ingestion_result["good_count"]

print("\nExpected good count:", expected_good_count)
print("Written Bronze count:", bronze_count)

if bronze_count == expected_good_count:
    print(
        "\nBronze count validation: PASSED"
    )
else:
    raise ValueError(
        "Bronze count validation failed. "
        f"Expected {expected_good_count}, "
        f"but found {bronze_count}."
    )


# ===============

Bronze batch path: hdfs:///capstone/warehouse/bronze/sales_order_lines/batch_id=api-00002-malformed-03/file_00000

Bronze record count: 3

Bronze schema:
root
 |-- id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_uom_qty: double (nullable = true)
 |-- price_unit: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- price_subtotal: double (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- write_date: timestamp (nullable = true)
 |-- qty_delivered: double (nullable = true)
 |-- qty_invoiced: double (nullable = true)
 |-- write_date_partition: date (nullable = true)


Bronze data preview:
+-------+---------------------+----------------------------------------------------------+---------------+----------+--------+--------------+-------------------+-------------------+-------------+------------+--------------------+
|id     |order_id             |product_id                         

In [12]:
# Complete Audit trail

# ============================================================
# LOAD ALL AUDIT EVENTS FOR THE CURRENT RUN
# ============================================================

CURRENT_RUN_ID = ingestion_result["run_id"]

run_audit_df = (
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == CURRENT_RUN_ID
    )
    .orderBy(
        F.col("created_at").asc()
    )
)


# ============================================================
# SHOW AUDIT EVENT SUMMARY
# ============================================================

print("Audit trail for run:", CURRENT_RUN_ID)

run_audit_df.select(
    "event_type",
    "status",
    "source_count",
    "good_count",
    "bad_count",
    "written_good_count",
    "written_bad_count",
    "created_at"
).show(
    truncate=False
)


# ============================================================
# COUNT AUDIT EVENTS BY TYPE AND STATUS
# ============================================================

print("\nAudit-event counts:")

run_audit_df.groupBy(
    "event_type",
    "status"
).count().orderBy(
    "event_type",
    "status"
).show(
    truncate=False
)


# ============================================================
# CHECK FOR PROCESSING ERRORS
# ============================================================

processing_errors_df = (
    run_audit_df
    .filter(
        F.col("event_type") == "PROCESSING_ERROR"
    )
)

processing_error_count = processing_errors_df.count()

print(
    "\nProcessing error count:",
    processing_error_count
)

if processing_error_count > 0:
    print("\nProcessing errors:")

    processing_errors_df.select(
        "source_file",
        "status",
        "error_message",
        "created_at"
    ).show(
        truncate=False
    )
else:
    print(
        "Processing-error validation: PASSED"
    )


# ============================================================
# CHECK CORRUPT-RECORD AUDIT COUNT
# ============================================================

corrupt_audit_count = (
    run_audit_df
    .filter(
        F.col("event_type") == "CORRUPT_RECORD"
    )
    .count()
)

expected_bad_count = ingestion_result["bad_count"]

print(
    "\nExpected bad-record count:",
    expected_bad_count
)

print(
    "Corrupt records in audit table:",
    corrupt_audit_count
)

if corrupt_audit_count == expected_bad_count:
    print(
        "Corrupt-record audit validation: PASSED"
    )
else:
    raise ValueError(
        "Corrupt-record audit count does not match "
        "the ingestion bad-record count."
    )


# ============================================================
# VERIFY RECONCILIATION STATUS
# ============================================================

reconciliation_df = (
    run_audit_df
    .filter(
        F.col("event_type") == "RECONCILIATION"
    )
)

failed_reconciliation_count = (
    reconciliation_df
    .filter(
        F.col("status") != "RECONCILED"
    )
    .count()
)

if failed_reconciliation_count == 0:
    print(
        "Reconciliation audit validation: PASSED"
    )
else:
    print(
        "Reconciliation audit validation: FAILED"
    )

    reconciliation_df.show(
        truncate=False,
        vertical=True
    )

Audit trail for run: a9f90d91-19c1-40d6-82ed-5ab81ca462f7
+--------------+---------------+------------+----------+---------+------------------+-----------------+--------------------------+
|event_type    |status         |source_count|good_count|bad_count|written_good_count|written_bad_count|created_at                |
+--------------+---------------+------------+----------+---------+------------------+-----------------+--------------------------+
|SCHEMA_DRIFT  |DRIFT_FOUND    |4           |3         |1        |NULL              |NULL             |2026-10-11 11:53:04.911545|
|CORRUPT_RECORD|PARSER_REJECTED|NULL        |NULL      |NULL     |NULL              |NULL             |2026-10-11 11:53:07.718343|
|RECONCILIATION|RECONCILED     |4           |3         |1        |3                 |1                |2026-10-11 11:53:11.448597|
|BATCH_SUMMARY |SUCCESS        |4           |3         |1        |3                 |1                |2026-10-11 11:53:12.076152|
+--------------+---------

In [13]:
# ============================================================
# RELOAD THE UPDATED SCHEMA-DRIFT TEMPLATE
# ============================================================

import importlib
import data_profilers.schema_drift as schema_drift

schema_drift = importlib.reload(
    schema_drift
)

process_landing_csv = (
    schema_drift.process_landing_csv
)

compare_schemas = (
    schema_drift.compare_schemas
)

print(
    "Reloaded template from:",
    schema_drift.__file__
)


# ============================================================
# TEST EXPECTED VS ACTUAL SCHEMA
# ============================================================

actual_schema = T.StructType([
    T.StructField(
        field.name,
        (
            T.IntegerType()
            if field.name == "id"
            else field.dataType
        ),
        field.nullable
    )
    for field in expected_schema.fields
])

comparison_result = compare_schemas(
    expected_schema,
    actual_schema
)

print("\nSchema comparison result:")

print(
    json.dumps(
        comparison_result,
        indent=4
    )
)


# ============================================================
# VALIDATE COMPATIBLE TYPE WIDENING
# ============================================================

assert (
    comparison_result["drift_detected"]
    is False
), (
    "Compatible int-to-bigint difference "
    "was incorrectly marked as drift."
)

assert (
    len(
        comparison_result[
            "compatible_type_changes"
        ]
    ) == 1
), (
    "Expected one compatible type difference."
)

print(
    "\nCompatible widening validation: PASSED"
)

Reloaded template from: /mnt/c/santhosh/capstone/product_analysis/data_profilers/schema_drift.py

Schema comparison result:
{
    "drift_detected": false,
    "added_columns": [],
    "missing_columns": [],
    "type_changes": [],
    "compatible_type_changes": [
        {
            "column": "id",
            "expected": "bigint",
            "actual": "int"
        }
    ]
}

Compatible widening validation: PASSED


In [14]:
# ============================================================
# INSPECT CORRUPT-RECORD IDENTIFIERS
# ============================================================

(
    spark.table(AUDIT_TABLE)
    .filter(
        (F.col("run_id") == ingestion_result["run_id"])
        & (F.col("event_type") == "CORRUPT_RECORD")
    )
    .select(
        "run_id",
        "batch_id",
        "source_path",
        "source_file",
        "target_path",
        "raw_record",
        "status"
    )
    .show(
        truncate=False,
        vertical=True
    )
)

print("SOURCE_FILE passed to template:")
print(SOURCE_FILE)

print("\nFile reported in returned result:")
print(
    ingestion_result["files"][0]["source_file"]
)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 run_id      | a9f90d91-19c1-40d6-82ed-5ab81ca462f7                                                                                                                                                                                                                                                                                                       
 batch_id    | api-00002-malformed-03                                                                                                                                                                                                                                                                             